In [8]:
import os
import zipfile

def compress_folder_with_level(folder_path, output_zip_prefix, chunk_size_gb=1, compress_level=6):
    chunk_size_bytes = chunk_size_gb * 1024 * 1024 * 1024
    if not os.path.exists(output_zip_prefix):
        os.makedirs(output_zip_prefix)
    zip_file_path = f"{output_zip_prefix}/compressed.zip"

    with zipfile.ZipFile(zip_file_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=compress_level) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

    with open(zip_file_path, 'rb') as f:
        part_num = 0
        while True:
            data = f.read(chunk_size_bytes)
            if not data:
                break
            part_num += 1
            part_file_name = f"{output_zip_prefix}/compressed.z{str(part_num).zfill(2)}"
            with open(part_file_name, 'wb') as part_file:
                part_file.write(data)

    os.remove(zip_file_path)
    print(f"Compressing {part_num} sub file.")

compress_folder_with_level("/home/snt/projects_lujun/agentCLS/script/datasets", "compressed_folder", chunk_size_gb=1, compress_level=9)


Compressing 1 sub file.


In [ ]:
def decompress_split_zip(output_zip_prefix, extract_to_folder):
    
    merged_zip_file = f"{output_zip_prefix}_merged.zip"
    with open(merged_zip_file, 'wb') as merged:
        part_num = 1
        while True:
            part_file_name = f"{output_zip_prefix}/compressed.z{str(part_num).zfill(2)}"
            if not os.path.exists(part_file_name):
                break
            with open(part_file_name, 'rb') as part_file:
                merged.write(part_file.read())
            part_num += 1

    with zipfile.ZipFile(merged_zip_file, 'r') as zipf:
        zipf.extractall(extract_to_folder)

    os.remove(merged_zip_file)
    print(f"{extract_to_folder} decompressed from {merged_zip_file}.")

decompress_split_zip("compressed_folder", "reconstructed_folder")


reconstructed_folder decompressed from compressed_folder_merged.zip.
